<a href="https://colab.research.google.com/github/doomguy0991/N132/blob/main/CS231n_Lecture3_Study_Notes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Setting up Colab environment...")
    repo_url = "https://github.com/doomguy0991/N132.git"
    if not os.path.exists("/content/N132"):
        !git clone $repo_url /content/N132
    %cd "/content/N132"
    print("Setup complete.")
else:
    print("Running locally. No setup needed.")


# CS231n Lecture 3 Study Notes: Regularization and Optimization

This Jupyter Notebook serves as a comprehensive, standalone study guide for **CS231n Lecture 3: Regularization and Optimization**. It is designed to provide a deep, university-level understanding of how we evaluate models, prevent overfitting, and navigate the loss landscape to find optimal parameters.

## Table of Contents

- [Section 0: Prerequisites & Recap](#section-0-prerequisites--recap)
    - [0.1 The Image Classification Setup](#01-the-image-classification-setup)
    - [0.2 The Linear Score Function](#02-the-linear-score-function)
    - [0.3 Data Loss Recap (Softmax & Cross-Entropy)](#03-data-loss-recap-softmax--cross-entropy)
- [Section 1: Loss Functions and Regularization](#section-1-loss-functions-and-regularization)
    - [1.1 The Crucial Missing Piece: Data Loss vs. Generalization](#11-the-crucial-missing-piece-data-loss-vs-generalization)
    - [1.2 The Generalization Problem & Occam's Razor](#12-the-generalization-problem--occams-razor)
    - [1.3 Regularization: Penalizing Complexity](#13-regularization-penalizing-complexity)
    - [1.4 L2 Regularization: Encouraging Spread-Out Weights](#14-l2-regularization-encouraging-spread-out-weights)
    - [1.5 L1 Regularization: Encouraging Sparsity](#15-l1-regularization-encouraging-sparsity)
    - [1.6 NumPy & PyTorch Implementation from Scratch](#16-numpy--pytorch-implementation-from-scratch)
    - [1.7 Summary & Key Takeaways](#17-summary--key-takeaways)
- [Section 2: Introduction to Optimization](#section-2-introduction-to-optimization)
    - [2.1 The Loss Landscape](#21-the-loss-landscape)
    - [2.2 Random Search: A Brute Force Failure](#22-random-search-a-brute-force-failure)
    - [2.3 Following the Slope: The Gradient](#23-following-the-slope-the-gradient)
    - [2.4 Numerical vs. Analytic Gradients](#24-numerical-vs-analytic-gradients)
    - [2.5 Numerical Gradient & Gradient Check Implementation](#25-numerical-gradient--gradient-check-implementation)
    - [2.6 Summary & Transition to Gradient Descent](#26-summary--transition-to-gradient-descent)
- [Section 3: Gradient Descent and its Variants](#section-3-gradient-descent-and-its-variants)
    - [3.1 Vanilla Gradient Descent](#31-vanilla-gradient-descent)
    - [3.2 Stochastic Gradient Descent (SGD)](#32-stochastic-gradient-descent-sgd)
    - [3.3 Pitfalls of Vanilla SGD](#33-pitfalls-of-vanilla-sgd)
    - [3.4 SGD with Momentum](#34-sgd-with-momentum)
    - [3.5 RMSProp: Adaptive Learning Rates](#35-rmsprop-adaptive-learning-rates)
    - [3.6 Adam: Combining Momentum and RMSProp](#36-adam-combining-momentum-and-rmsprop)
    - [3.7 Decoupled Weight Decay: Adam vs. AdamW](#37-decoupled-weight-decay-adam-vs-adamw)
    - [3.8 NumPy Optimizers Implementation from Scratch](#38-numpy-optimizers-implementation-from-scratch)
    - [3.9 Summary & Transition to Schedulers](#39-summary--transition-to-schedulers)
- [Section 4: Learning Rate Scheduling](#section-4-learning-rate-scheduling)
    - [4.1 Importance of the Learning Rate $\alpha$](#41-importance-of-the-learning-rate-alpha)
    - [4.2 Schedulers (Step Decay, Cosine Decay, Warmup)](#42-schedulers-step-decay-cosine-decay-warmup)
    - [4.3 NumPy Implementation of Schedulers](#43-numpy-implementation-of-schedulers)
    - [4.4 Batch Size & The Linear Scaling Law](#44-batch-size--the-linear-scaling-law)
- [Section 5: Second-Order Optimization (Brief Overview)](#section-5-second-order-optimization-brief-overview)
    - [5.1 Second-Order Approximations & Newton's Method](#51-second-order-approximations--newtons-method)
    - [5.2 Why Newton's Method Fails in Deep Learning](#52-why-newtons-method-fails-in-deep-learning)
    - [5.3 Final Summary & Lecture Takeaways](#53-final-summary--lecture-takeaways)


# Section 0: Prerequisites & Recap

Before diving into the core concepts of regularization and optimization, let's briefly review the foundational components of the parametric image classification pipeline established in the previous lecture.

## 0.1 The Image Classification Setup
In image classification, our goal is to map an input image $x_i$ to a single categorical label $y_i$ from a predefined label set of size $C$ (e.g., $C=10$ for CIFAR-10).
*   **Input Representation**: A color image of height $H$, width $W$, and $3$ color channels (Red, Green, Blue) is flattened into a 1D feature vector $x_i \in \mathbb{R}^D$, where $D = H \times W \times 3$.
*   For a standard CIFAR-10 image of size $32 \times 32 \times 3$, the input size is $D = 3072$.

## 0.2 The Linear Score Function
The **linear classifier** is a parametric model that computes scores for each of the $C$ classes using a matrix multiplication and a bias vector shift:

$$ f(x_i; W, b) = W x_i + b $$

Where:
*   $x_i \in \mathbb{R}^{D \times 1}$ is the input image vector.
*   $W \in \mathbb{R}^{C \times D}$ is the weight matrix (representing learned templates for each class).
*   $b \in \mathbb{R}^{C \times 1}$ is the bias vector (representing class-specific prior offsets).
*   $f(x_i; W, b) \in \mathbb{R}^{C \times 1}$ is a vector of raw scores (also called **logits**) for each class.

## 0.3 Data Loss Recap (Softmax & Cross-Entropy)
To measure how "bad" our scores are compared to the ground-truth labels, we define a **data loss function**. For classification, the standard choice is the **Softmax Loss** (cross-entropy loss), which interprets raw scores as normalized log probabilities.

For a single training example $i$, with scores $s = f(x_i; W, b)$, the probability of the correct class $y_i$ is computed as:

$$ P(Y = y_i \mid X = x_i) = \frac{e^{s_{y_i}}}{\sum_j e^{s_j}} $$

The loss $L_i$ is the negative log-likelihood of the correct class:

$$ L_i = -\log \left( \frac{e^{s_{y_i}}}{\sum_j e^{s_j}} \right) $$

The average data loss over the entire dataset of $N$ training examples is:

$$ L_{data} = \frac{1}{N} \sum_{i=1}^N L_i $$


# Section 1: Loss Functions and Regularization

## 1.1 The Crucial Missing Piece: Data Loss vs. Generalization

In the recap above, we defined $L_{data}$ to measure how well our model fits the training dataset. If our only goal were to minimize $L_{data}$, we would guide our model to fit every single training data point perfectly. 

However, this leads to a fundamental machine learning pathology: **Overfitting**.

> [!IMPORTANT]
> **Data Loss Fallacy:** A model that achieves $0$ data loss on the training set is not necessarily a good model. In fact, it is often a very poor model because it has memorized the noise, background clutter, and idiosyncrasies of the training data instead of learning the underlying semantic concepts.
>
> The ultimate goal of machine learning is **generalization**—how well the model performs on new, completely unseen data (the test set).

## 1.2 The Generalization Problem & Occam's Razor

Consider a classic curve-fitting task. We want to fit a function $y = f(x)$ to a set of noisy 2D points:

```
                  * (Noisy training point)
          *
    *          *
        *
```

*   **Model 1 ($f_1$):** A simple linear or low-degree polynomial curve that passes near the points but doesn't hit them perfectly.
*   **Model 2 ($f_2$):** A high-degree polynomial that bends, twists, and oscillates wildly to pass exactly through every single noisy training point.

Model $f_2$ has $0$ training error. But if we draw new points from the same underlying distribution, Model $f_1$ (the simpler model) will have a much lower error than Model $f_2$. 

This is the principle of **Occam's Razor**:
> "Numquam ponenda est pluralitas sine necessitate" (Plurality should not be posited without necessity).
> In machine learning, this translates to: **If we have multiple competing hypotheses (models) that explain the training data, we should prefer the simplest one.**

Regularization is the mathematical tool we use to enforce Occam's Razor.


## 1.3 Regularization: Penalizing Complexity

To steer our model towards simpler hypotheses, we modify our loss function by adding a **regularization term** $R(W)$ that measures the complexity of the weights $W$.

The complete, joint loss function is:

$$ L(W) = \underbrace{\frac{1}{N} \sum_{i=1}^N L_i(f(x_i; W), y_i)}_{\text{Data Loss (Fit the data)}} + \underbrace{\lambda R(W)}_{\text{Regularization Loss (Keep weights simple)}} $$

Where:
*   $L_i$ is the data loss for example $i$ (e.g., Softmax loss).
*   $R(W)$ is the regularization function that penalizes weight complexity. Note that $R(W)$ depends *only* on the weight parameters $W$, not on the data $x$ or labels $y$.
*   $\lambda$ is the **regularization strength** (a crucial hyperparameter).

### The Tug-of-War Intuition
The loss function sets up a mathematical tug-of-war between two opposing forces:
1.  **Data Loss** tries to make the model fit the training data as accurately as possible (encouraging complex, highly-specific parameters if needed).
2.  **Regularization Loss** tries to drive the parameters towards $0$ or simplify them, preventing the model from fitting the training data too closely.

By adjusting $\lambda$, we control the balance:
*   $\lambda = 0$: No regularization. The model is free to overfit completely.
*   $\lambda \to \infty$: Extreme regularization. The model ignores the data entirely and sets weights to $0$, resulting in underfitting.


### Pipeline / Objective Balance Diagram

![Diagram 1](assets/lecture3_diagram_1.png)


## 1.4 L2 Regularization: Encouraging Spread-Out Weights

The most common form of regularization is **L2 Regularization** (also known as **weight decay** or **Ridge regression**).

### Mathematical Definition
L2 regularization penalizes the squared Euclidean norm of the weight matrix:

$$ R(W) = \sum_{k} \sum_{l} W_{k, l}^2 = \|W\|_F^2 $$

Where $\|W\|_F$ is the **Frobenius Norm** of the matrix (the matrix equivalent of the vector L2 norm).

### The "Spread-Out" Intuition
Why does L2 regularization prefer simpler models? Let's trace through the lecture's elegant dot-product example.

Suppose we have an input vector $x = [1, 1, 1, 1]^T$ and we compare two different weight vectors:
*   $W_1 = [1, 0, 0, 0]^T$
*   $W_2 = [0.25, 0.25, 0.25, 0.25]^T$

Let's compute the raw score (dot product) for both:
*   $W_1^T x = (1 \cdot 1) + (0 \cdot 1) + (0 \cdot 1) + (0 \cdot 1) = 1$
*   $W_2^T x = (0.25 \cdot 1) + (0.25 \cdot 1) + (0.25 \cdot 1) + (0.25 \cdot 1) = 1$

Both weight vectors produce the *exact same score* of $1$. Under the data loss term alone, they are completely equivalent. 

Now, let's compute the L2 regularization penalty $R(W) = \|W\|_2^2$ for both:
*   $R(W_1) = 1^2 + 0^2 + 0^2 + 0^2 = \mathbf{1.0}$
*   $R(W_2) = 0.25^2 + 0.25^2 + 0.25^2 + 0.25^2 = 0.0625 + 0.0625 + 0.0625 + 0.0625 = \mathbf{0.25}$

**Key Result:** The L2 regularization penalty for $W_1$ is **four times larger** than for $W_2$! Thus, the optimizer will strongly prefer $W_2$ over $W_1$.

### Why is a spread-out weight vector "simpler" and better?
*   **Pixel Dependency**: $W_1$ relies entirely on a single input pixel to make its prediction. If that pixel becomes noisy, obscured, or corrupted, the prediction will fail completely.
*   **Generalization**: $W_2$ distributes its weights evenly across all input pixels. It makes a collective, diffuse decision. This is highly robust to noise and local pixel variations, leading to far better generalization on unseen images.
*   **Geometric effect**: By squaring the terms, L2 places a heavily disproportionate penalty on large, individual weights, pushing the weights to be small and diffuse.


## 1.5 L1 Regularization: Encouraging Sparsity

Another fundamental regularization technique is **L1 Regularization** (also known as **Lasso**).

### Mathematical Definition
L1 regularization penalizes the sum of the absolute values of the weight matrix elements:

$$ R(W) = \sum_{k} \sum_{l} |W_{k, l}| $$

### The "Sparsity" Intuition
Unlike L2, which distributes weight values evenly, L1 regularization drives many weights to be **exactly zero**. A vector with many zero entries is called a **sparse vector**.

Let's re-evaluate our earlier vectors $W_1$ and $W_2$ under the L1 penalty:
*   $W_1 = [1, 0, 0, 0]^T \implies R(W_1) = |1| + |0| + |0| + |0| = \mathbf{1.0}$
*   $W_2 = [0.25, 0.25, 0.25, 0.25]^T \implies R(W_2) = |0.25| + |0.25| + |0.25| + |0.25| = \mathbf{1.0}$

Under L1 regularization, $W_1$ and $W_2$ incur the *exact same penalty*! 

So, what makes L1 drive weights to zero?
Consider the derivative of the regularization terms:
*   For L2: $\frac{\partial}{\partial w} (w^2) = 2w$. As $w$ approaches $0$, the derivative (the force pulling $w$ down) also approaches $0$. The pull gets weaker and weaker.
*   For L1: $\frac{\partial}{\partial w} (|w|) = \text{sign}(w) = \pm 1$ (for $w \neq 0$). The force pulling $w$ down is constant ($1$ or $-1$) no matter how small $w$ is. It will pull it all the way to $0$ and lock it there.

### Geometric Interpretation

If we visualize the constraint surfaces in 2D space:
*   **L2 Constraint ($\|W\|_2^2 \le C$)** forms a **circle**. The optimization path is likely to hit the circle anywhere, typically leading to small, non-zero coordinates.
*   **L1 Constraint ($\|W\|_1 \le C$)** forms a **diamond** (with sharp corners along the coordinate axes). The optimization path is highly likely to hit the diamond at one of its corners, where one of the coordinates is exactly $0$.

```
       L2 (Circle)               L1 (Diamond)
          .---.                       /\
         /     \                     /  \
        |   x   |                   /  x \
         \     /                    \    /
          '---'                      \  /
                                      \/
```

### Why use L1 vs L2?
*   **Use L2** when you want to use all your features robustly and expect a diffuse distribution of information (default for almost all deep learning architectures).
*   **Use L1** when you want **feature selection**. By forcing weights to exactly $0$, L1 identifies the most critical pixels/features and discards the rest. This creates highly interpretable, lightweight models.


## 1.6 NumPy & PyTorch Implementation from Scratch

Below, we implement both the L1/L2 weight penalties and verify the calculations programmatically. We also show how to compute the joint (Softmax + Regularization) loss function.

In [ ]:
import numpy as np

# --- Verification of L1 and L2 weight penalty intuition ---
x = np.array([1.0, 1.0, 1.0, 1.0])
w1 = np.array([1.0, 0.0, 0.0, 0.0])
w2 = np.array([0.25, 0.25, 0.25, 0.25])

# 1. Compute scores
score1 = np.dot(w1, x)
score2 = np.dot(w2, x)

# 2. Compute L2 Regularization penalty
l2_w1 = np.sum(np.square(w1))
l2_w2 = np.sum(np.square(w2))

# 3. Compute L1 Regularization penalty
l1_w1 = np.sum(np.abs(w1))
l1_w2 = np.sum(np.abs(w2))

print("--- L1 vs L2 Weight Penalty Verification ---")
print(f"w1.x = {score1:.4f} | w2.x = {score2:.4f} (Equal Data Loss)")
print(f"L2 penalty for w1 (concentrated): {l2_w1:.4f}")
print(f"L2 penalty for w2 (spread-out): {l2_w2:.4f} (4x lower penalty!)")
print(f"L1 penalty for w1 (concentrated): {l1_w1:.4f}")
print(f"L1 penalty for w2 (spread-out): {l1_w2:.4f} (Exactly equal!)")


# --- NumPy joint loss computation ---
def compute_softmax_loss(scores, y):
    """
    Computes Softmax loss (cross-entropy) for a batch of scores.
    scores: shape (N, C)
    y: shape (N,) containing ground-truth indices
    """
    # Numerically stable softmax
    shifted_scores = scores - np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(shifted_scores)
    probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
    
    N = scores.shape[0]
    correct_probs = probs[np.arange(N), y]
    loss_data = -np.log(correct_probs + 1e-15)
    return np.mean(loss_data), probs

# Let's mock a batch of 3 CIFAR-10 images
np.random.seed(42)
N, C, D = 3, 10, 3072
X_mock = np.random.randn(N, D)
y_mock = np.array([2, 5, 0]) # Correct class labels

# Define model weights and biases
W_mock = np.random.randn(C, D) * 0.01
b_mock = np.zeros(C)

# Forward pass
scores_mock = X_mock.dot(W_mock.T) + b_mock

# Compute losses
loss_data, probs_mock = compute_softmax_loss(scores_mock, y_mock)
reg_strength = 0.1

loss_reg_l2 = reg_strength * np.sum(np.square(W_mock))
loss_reg_l1 = reg_strength * np.sum(np.abs(W_mock))

total_loss_l2 = loss_data + loss_reg_l2
total_loss_l1 = loss_data + loss_reg_l1

print("\n--- NumPy Full Loss Computation ---")
print(f"Data Loss (Softmax): {loss_data:.6f}")
print(f"L2 Penalty: {loss_reg_l2:.6f} | Total L2 Loss: {total_loss_l2:.6f}")
print(f"L1 Penalty: {loss_reg_l1:.6f} | Total L1 Loss: {total_loss_l1:.6f}")


In [ ]:
import torch
import torch.nn as nn

# --- PyTorch implementation --- 
# Note: PyTorch has built-in L2 decay inside its optimizers (via weight_decay=lambda).
# However, we can compute L1 and L2 manually directly on model parameters.

class LinearClassifier(nn.Module):
    def __init__(self, in_features=3072, out_features=10):
        super().__init__()
        self.fc = nn.Linear(in_features, out_features)
        
    def forward(self, x):
        return self.fc(x)

# Initialize model
torch.manual_seed(42)
model = LinearClassifier()
X_torch = torch.randn(3, 3072)
y_torch = torch.tensor([2, 5, 0], dtype=torch.long)

# Forward pass
logits = model(X_torch)

# Data Loss
criterion = nn.CrossEntropyLoss()
loss_data_torch = criterion(logits, y_torch)

# Manual L2 & L1 Regularization
lambda_reg = 0.01
l2_reg_loss = torch.tensor(0.0)
l1_reg_loss = torch.tensor(0.0)

for name, param in model.named_parameters():
    # Standard convention: Regularize weights (fc.weight), ignore biases (fc.bias)
    if 'weight' in name:
        l2_reg_loss += torch.sum(param ** 2)
        l1_reg_loss += torch.sum(torch.abs(param))

total_l2_loss_torch = loss_data_torch + lambda_reg * l2_reg_loss
total_l1_loss_torch = loss_data_torch + lambda_reg * l1_reg_loss

print("--- PyTorch Loss Computation ---")
print(f"Data Loss (Cross-Entropy): {loss_data_torch.item():.6f}")
print(f"L2 Penalty: {l2_reg_loss.item():.6f} | Total L2 Loss: {total_l2_loss_torch.item():.6f}")
print(f"L1 Penalty: {l1_reg_loss.item():.6f} | Total L1 Loss: {total_l1_loss_torch.item():.6f}")


---
### Summary & Key Takeaways

*   **Data Loss vs. Generalization:** Minimizing training loss alone leads to overfitting. The true target is high generalization performance on unseen data.
*   **Occam's Razor:** Prefer the simplest model that explains the data. We mathematically express "simplicity" via a regularization penalty $R(W)$ added to the objective.
*   **L2 Regularization (Ridge/Weight Decay):** Penalizes squared weight magnitudes ($\|W\|_F^2$). It favors diffuse, spread-out weight vectors that distribute dependency across all input pixels, rendering the model robust to noise.
*   **L1 Regularization (Lasso):** Penalizes absolute weight magnitudes ($\sum |W|$). It promotes **sparsity** by driving non-critical weights to exactly $0$, performing automatic feature selection.
*   **Tug-of-War:** The joint loss function features a balance between fitting training data (Data Loss) and keeping weights simple (Regularization Loss), controlled by the hyperparameter $\lambda$.

**Up Next:** Now that we have defined our model (Linear Classifier) and a rigorous method to evaluate its quality (Joint Loss = Softmax + Regularization), we face the ultimate question: **How do we find the weight matrix $W$ that minimizes this loss?** In the next section, we introduce the mathematics and mechanisms of **Optimization**.

# Section 2: Introduction to Optimization

In the previous section, we established the core mathematical goal of supervised machine learning: defining a joint loss function $L(W)$ that perfectly balances training data fit (Data Loss) and model simplicity (Regularization Loss). 

Now, we face the operational challenge: **How do we find the weight parameters $W$ that minimize this loss?** This is the domain of **Optimization**.

## 2.1 The Loss Landscape

To understand optimization, it is useful to visualize the **Loss Landscape**. 

Imagine you are standing on a rugged mountain range (such as the Appalachian or Swiss Alps).
*   Your physical location (longitude and latitude) represents the **weights** $W$ of your model. In a real neural network, this space is not 2-dimensional, but millions or billions of dimensions.
*   Your altitude (height above sea level) represents the **loss** $L(W)$ of your model.
*   Your goal is to reach the absolute lowest point in the landscape—the deepest valley (**Global Minimum**).

### The Blindfolded Hiker Analogy
The core challenge of machine learning optimization is that you are **blindfolded**. You cannot look down from a helicopter to see where the valley is. You have no visual information about the surrounding mountains. 
*   You can only feel the slope of the ground directly underneath your feet.
*   You must make step-by-step decisions about which direction to walk based purely on local terrain features.

## 2.2 Random Search: A Brute Force Failure

A naive first attempt at optimization is **Random Search**. We simply try a large number of random weight matrices, evaluate their loss on the training data, and select the one that yields the lowest loss.

Let's implement this naive strategy using NumPy and evaluate its performance on a mock dataset.

In [ ]:
import numpy as np

# Let's simulate a random search optimization for our mock classification task
np.random.seed(42)
N, C, D = 100, 10, 3072 # 100 images, 10 classes, CIFAR-10 size
X_train = np.random.randn(N, D)
y_train = np.random.randint(0, C, N)

def evaluate_loss(W, X, y):
    scores = X.dot(W.T)
    shifted_scores = scores - np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(shifted_scores)
    probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
    correct_probs = probs[np.arange(X.shape[0]), y]
    return -np.mean(np.log(correct_probs + 1e-15))

# Brute-force random search
best_loss = float('inf')
best_W = None

num_trials = 1000
for trial in range(num_trials):
    # Sample random weights
    W_trial = np.random.randn(C, D) * 0.001
    loss = evaluate_loss(W_trial, X_train, y_train)
    
    if loss < best_loss:
        best_loss = loss
        best_W = W_trial

# Evaluate a baseline random weight matrix to see the starting point
random_loss = evaluate_loss(np.random.randn(C, D) * 0.001, X_train, y_train)
print("--- Random Search Evaluation ---")
print(f"Baseline Random Loss: {random_loss:.6f}")
print(f"Best Loss found by Random Search in {num_trials} trials: {best_loss:.6f}")


## 2.3 Following the Slope: The Gradient

Instead of blindly guessing random directions, we can use the local slope of the landscape to guide our steps. In calculus, this slope is represented by the **Derivative** (in 1D) or the **Gradient** (in multiple dimensions).

### Mathematical Definition
In 1D, the derivative of a function $f(x)$ at point $x$ is defined as the limit:

$$ \frac{df(x)}{dx} = \lim_{h \to 0} \frac{f(x + h) - f(x)}{h} $$

When we have a function that takes a vector or matrix of parameters $W$, the gradient $\nabla_W L$ (read as "nabla $W$ of $L$") is a vector of **partial derivatives** for each parameter:

$$ \nabla_W L = \left[ \frac{\partial L}{\partial W_{1,1}}, \frac{\partial L}{\partial W_{1,2}}, \dots, \frac{\partial L}{\partial W_{C,D}} \right] $$

> [!NOTE]
> **Direction of Steepest Descent:**
> *   The gradient vector $\nabla_W L$ points in the direction of **steepest increase** in loss (up the hill).
> *   Therefore, the **negative gradient** $-\nabla_W L$ points in the direction of **steepest decrease** in loss (down the hill).
>
> This is the fundamental premise of all gradient-based learning: **we should iteratively update our weights by taking a small step in the direction of the negative gradient.**

## 2.4 Numerical vs. Analytic Gradients

There are two distinct ways to compute the gradient of a loss function:

### 1. Numerical Gradient (Approximate and Slow)
We use the finite difference limit approximation directly. For each individual weight parameter, we add a very small value $h$ (e.g., $10^{-5}$), evaluate the new loss, and compute the slope:

$$ \frac{\partial L}{\partial W_{j,k}} \approx \frac{L(W + h \cdot E_{j,k}) - L(W)}{h} $$

where $E_{j,k}$ is a matrix with $1$ at index $(j,k)$ and $0$ everywhere else.

*   **Pros:** Incredibly easy to write and verify. It requires no calculus knowledge.
*   **Cons:** Extremely computationally expensive. If our weight matrix has $30,720$ parameters, we must compute the full loss function $30,720$ times just to take a *single* step! This is completely unfeasible for deep learning.

### 2. Analytic Gradient (Exact and Fast)
We use calculus (the chain rule, derivative rules) to derive a direct mathematical formula for the gradient. 

*   **Pros:** Mathematically exact and incredibly fast to compute. We can compute the gradients for all millions of parameters in a single forward/backward pass.
*   **Cons:** Highly error-prone to write by hand. A tiny algebraic mistake (e.g., a missing minus sign or transposed matrix) will result in incorrect updates, causing the model to fail to train.


### Gradient Computation & Checking Flow

![Diagram 2](assets/lecture3_diagram_2.png)


## 2.5 Numerical Gradient & Gradient Check Implementation

Below, we implement the numerical gradient algorithm from scratch and run a **Gradient Check** against a toy function to verify its precision.

In [ ]:
import numpy as np

def compute_numerical_gradient(loss_fn, W, h=1e-5):
    """
    Computes the numerical gradient of loss_fn at W using finite differences.
    """
    grad = np.zeros_like(W)
    
    # Iterate over all elements of the weight matrix
    it = np.nditer(W, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        ix = it.multi_index
        old_value = W[ix]
        
        # 1. Evaluate loss at W + h
        W[ix] = old_value + h
        loss_plus = loss_fn(W)
        
        # 2. Reset W to original value
        W[ix] = old_value
        
        # 3. Compute derivative
        grad[ix] = (loss_plus - loss_fn(W)) / h
        
        it.iternext()
        
    return grad

# Let's perform a simple "Gradient Check" to verify our implementation.
# We will define a toy loss function L(W) = sum(W^2) which has an obvious analytic gradient: dL/dW = 2 * W.
W_toy = np.array([[1.0, 2.0], [3.0, 4.0]])
toy_loss_fn = lambda W: np.sum(np.square(W))

grad_num = compute_numerical_gradient(toy_loss_fn, W_toy)
grad_analytic = 2 * W_toy

# Compare the two using relative error formula
rel_error = np.abs(grad_num - grad_analytic) / np.maximum(np.abs(grad_num) + np.abs(grad_analytic), 1e-8)

print("--- Gradient Check Verification ---")
print("Numerical Gradient:\n", grad_num)
print("Analytic Gradient:\n", grad_analytic)
print("Max Relative Error:", np.max(rel_error))


## 2.6 Summary & Transition to Gradient Descent

*   **Landscape Navigation:** Optimization is the search for optimal parameter settings that minimize the loss landscape.
*   **Steepest Descent:** Gradients point in the direction of steepest loss ascent. Taking steps in the opposite direction ($-\nabla_W L$) drives the loss down.
*   **Approximating vs Deriving:**
    *   Numerical gradients are simple to implement but extremely slow ($O(D)$ evaluations for $D$ parameters).
    *   Analytic gradients are derived using calculus, exact, and extremely fast ($O(1)$ updates) but prone to algebraic bugs.
*   **Debugging with Gradient Checks:** We always write a slow numerical gradient block to check our fast analytic updates. If the relative error is $< 10^{-7}$, the analytical model is verified.

---

**Up Next:** Now that we have the gradient to show us the way, how do we use it to construct powerful optimization algorithms that navigate valleys and jump over mountain ridges? In **Section 3**, we implement **Gradient Descent** and explore modern algorithms like **SGD, Momentum, RMSProp, and Adam**.

# Section 3: Gradient Descent and its Variants

Having defined a loss function that evaluates weight configuration quality (Section 1) and the gradient vector that acts as a directional compass (Section 2), we now introduce the actual algorithms that perform the updates. 

We will explore **Vanilla Gradient Descent**, **Stochastic Gradient Descent (SGD)**, trace SGD's geological pitfalls, and build progressively sophisticated modern optimizers (**Momentum**, **RMSProp**, **Adam**, and **AdamW**).

## 3.1 Vanilla Gradient Descent

In **Vanilla (or Batch) Gradient Descent**, we evaluate the gradient of our loss function over the **entire training dataset** to perform a single weight update.

### Mathematical Formulation
$$ W_{t+1} = W_t - \alpha \nabla_W L(W_t) $$

Where:
*   $W_t$ is the current weight parameter matrix.
*   $\alpha$ (alpha) is the **learning rate** or step size. It determines how far we walk in the direction of the negative gradient.
*   $\nabla_W L(W_t)$ is the exact gradient evaluated across the entire training set.

### Conceptual Implementation
```python
# Vanilla Batch Gradient Descent Loop
while True:
    weights_grad = evaluate_gradient(loss_fn, data, weights)
    weights += - step_size * weights_grad
```

## 3.2 Stochastic Gradient Descent (SGD)

Evaluating the exact gradient over the entire dataset of size $N$ is highly computationally intensive, especially when $N$ contains millions of images (e.g., ImageNet). 

### The Mini-Batch Solution
Instead of waiting to process the whole dataset to take a single step, **Stochastic Gradient Descent (SGD)** samples a random small subset of training examples called a **mini-batch** (of size $B$, typically 32, 64, 128, or 256). 

We estimate the gradient purely based on this mini-batch:

$$ W_{t+1} = W_t - \alpha \nabla_W L_{batch}(W_t) $$

### Why does this work?
Because we sample mini-batches uniformly at random, the mini-batch gradient is an **unbiased estimator** of the true, full-dataset gradient:

$$ \mathbb{E}\left[ \nabla_W L_{batch}(W) \right] = \nabla_W L_{full}(W) $$

We get a noisy but mathematically valid approximation of the correct direction, allowing us to perform updates orders of magnitude faster.

*   **Epoch**: One complete pass through the entire training dataset (e.g., if $N=1000$ and batch size $B=100$, one epoch consists of 10 update steps).

## 3.3 Pitfalls of Vanilla SGD

Vanilla SGD performs terribly in complex loss landscapes due to three main geological pitfalls:

### 1. Poor Conditioning (Narrow Valleys / Ravines)
In many regions, the loss is highly sensitive to changes in one direction and highly insensitive to changes in another. This is represented by a **Hessian matrix** (second-order derivatives) with a very high condition number (large ratio between largest and smallest eigenvalues).
*   **SGD Behavior:** SGD will oscillate wildly back and forth across the steep, narrow walls of the valley, while making agonizingly slow progress along the flat bottom of the valley towards the minimum.

### 2. Local Minima
The loss landscape of a deep neural network is non-convex. It contains many **local minima**—valleys where the gradient is exactly $0$ but the loss is much higher than the global minimum. Once SGD enters a local minimum, it becomes trapped forever because the update term becomes $0$.

### 3. Saddle Points
A **saddle point** is a point where the gradient is $0$, but the landscape curves up in some directions and down in others (shaped like a horse saddle). 
*   **Curse of High Dimensions:** In multi-million dimensional spaces, local minima are actually very rare. For a point to be a local minimum, the landscape must curve upward along *every single* dimension. This is statistically highly unlikely. Instead, almost all flat regions in deep networks are saddle points.
*   **SGD Behavior:** Because the gradient is $0$ at the saddle center, SGD will slow to a complete crawl or get permanently stuck, ignoring the fact that escaping down the curving sides is possible.

## 3.4 SGD with Momentum

To solve the pitfalls of vanilla SGD, we introduce **Momentum**, drawing inspiration from Newtonian physics.

### Physics Intuition
Imagine a heavy iron ball rolling down the loss landscape.
*   Instead of resetting its step direction at every single point (like SGD), the ball builds up **velocity** as it rolls down steep inclines.
*   Its physical momentum carries it straight past small local minima and flat saddle points.
*   When it oscillates back and forth along narrow canyon walls, the high-frequency transverse velocities cancel each other out, while the longitudinal velocity along the valley bottom builds up, accelerating convergence.

### Mathematical Formulation

$$ \mathbf{v}_{t+1} = \rho \mathbf{v}_t + \nabla_W L(W_t) $$
$$ W_{t+1} = W_t - \alpha \mathbf{v}_{t+1} $$

Where:
*   $\mathbf{v}$ is the accumulated velocity vector (initialized to $0$).
*   $\rho$ (rho) is the **momentum decay coefficient** (or friction, typically set to $0.9$ or $0.99$). It controls how much velocity we retain from the previous step.
*   $\alpha$ is the learning rate.

## 3.5 RMSProp: Adaptive Learning Rates

While Momentum speeds up traversal by accumulating direction, **RMSProp** (Root Mean Square Propagation, proposed by Geoffrey Hinton) changes the *step size* dynamically for *each individual parameter*.

### Intuition
If a parameter has a consistently massive gradient, we want to decrease its effective learning rate to prevent wild oscillations. Conversely, if a parameter has a tiny gradient (flat landscape), we want to boost its step size to speed up progress.

### Mathematical Formulation

$$ \mathbf{s}_{t+1} = \beta_2 \mathbf{s}_t + (1 - \beta_2) \left( \nabla_W L(W_t) \right)^2 \quad \text{(Element-wise squaring)} $$
$$ W_{t+1} = W_t - \frac{\alpha}{\sqrt{\mathbf{s}_{t+1}} + \epsilon} \odot \nabla_W L(W_t) $$

Where:
*   $\mathbf{s}$ is a running average of the squared gradients (always positive, initialized to $0$).
*   $\beta_2$ is the decay hyperparameter (typically $0.99$ or $0.999$).
*   $\epsilon$ (epsilon) is a tiny smoothing term (e.g., $10^{-8}$) to prevent division by zero.
*   $\odot$ represents element-wise multiplication.

**Effect:** In steep directions ($
abla_W L$ is large), $\mathbf{s}$ becomes large, dividing the step by a large number and dampening oscillations. In flat directions ($
abla_W L$ is small), $\mathbf{s}$ becomes small, dividing by a tiny number and significantly boosting the step size.

## 3.6 Adam: Combining Momentum and RMSProp

**Adam** (Adaptive Moment Estimation) is the gold-standard optimizer in modern deep learning. It combines the accumulated velocity of **Momentum** (first moment) with the adaptive scaling of **RMSProp** (second moment).

### The Basic Equations
$$ \mathbf{m}_{t+1} = \beta_1 \mathbf{m}_t + (1 - \beta_1) \nabla_W L(W_t) \quad \text{(First Moment - Momentum)} $$
$$ \mathbf{v}_{t+1} = \beta_2 \mathbf{v}_t + (1 - \beta_2) \left( \nabla_W L(W_t) \right)^2 \quad \text{(Second Moment - RMSProp)} $$

### The Bias Correction Derivation

At $t=0$, $\mathbf{m}_0$ and $\mathbf{v}_0$ are initialized to $0$. Because $\beta_1$ and $\beta_2$ are close to $1$ (e.g., $0.9$ and $0.999$), the running estimates are heavily biased towards $0$ at the first few time steps. This results in a tiny second moment, dividing the step size by nearly zero and causing a catastrophic initial jump.

To resolve this, Adam applies **bias correction** to both moments.

> [!NOTE]
> **Mathematical Derivation of Bias Correction:**
> Let's unroll the recursive formula of $\mathbf{m}_t$ from $t=0$:
>
> $$ \mathbf{m}_t = (1 - \beta_1) \sum_{i=1}^t \beta_1^{t-i} \mathbf{g}_i $$
>
> where $\mathbf{g}_i = \nabla_W L(W_{i-1})$. Taking the mathematical expectation $\mathbb{E}[\cdot]$ on both sides, assuming the gradients $\mathbf{g}_i$ come from a stationary distribution with constant mean $\mathbb{E}[\mathbf{g}_i] = \mathbf{g}$:
>
> $$ \mathbb{E}[\mathbf{m}_t] = \mathbb{E}\left[ (1 - \beta_1) \sum_{i=1}^t \beta_1^{t-i} \mathbf{g}_i \right] = (1 - \beta_1) \sum_{i=1}^t \beta_1^{t-i} \mathbb{E}[\mathbf{g}_i] $$
> $$ \mathbb{E}[\mathbf{m}_t] = \mathbf{g} (1 - \beta_1) \sum_{i=1}^t \beta_1^{t-i} $$
>
> The summation is a finite geometric series $\sum_{j=0}^{t-1} \beta_1^j = \frac{1 - \beta_1^t}{1 - \beta_1}$. Substituting this in:
>
> $$ \mathbb{E}[\mathbf{m}_t] = \mathbf{g} (1 - \beta_1) \cdot \frac{1 - \beta_1^t}{1 - \beta_1} = \mathbf{g} (1 - \beta_1^t) $$
>
> Therefore, to get an unbiased estimator of the true gradient mean $\mathbf{g}$, we must divide $\mathbf{m}_t$ by $(1 - \beta_1^t)$:
>
> $$ \hat{\mathbf{m}}_t = \frac{\mathbf{m}_t}{1 - \beta_1^t} $$
>
> The exact same derivation applies to the second moment $\mathbf{v}_t$:
>
> $$ \hat{\mathbf{v}}_t = \frac{\mathbf{v}_t}{1 - \beta_2^t} $$

### The Final Adam Update
$$ W_{t+1} = W_t - \frac{\alpha}{\sqrt{\hat{\mathbf{v}}_{t+1}} + \epsilon} \odot \hat{\mathbf{m}}_{t+1} $$

## 3.7 Decoupled Weight Decay: Adam vs. AdamW

When we use **L2 Regularization** in machine learning, we add the penalty term $\lambda \|W\|^2_2$ to our loss. 

### The Adam L2 Conflict
In default Adam, this penalty is added directly to the loss *before* computing the gradient. This means the gradient sent to Adam is:

$$ \mathbf{g}_t = \nabla_W L_{data}(W_t) + 2 \lambda W_t $$

This gradient is then passed into the running moments $\mathbf{m}_t$ and $\mathbf{v}_t$. 
*   **The Issue:** Because we divide the update by $\sqrt{\mathbf{v}_t}$, weight coordinates that have historically massive data gradients will have their regularization penalty divided by a large value, regularizing them *less*.
*   Conversely, coordinate weights with near-zero data gradients will have their L2 decay heavily amplified.

This conflicts with the mathematical definition of weight decay, which should apply a constant, independent decay rate to all parameters.

### The AdamW Solution
**AdamW** (Decoupled Weight Decay, proposed by Loshchilov & Hutter) decouples the regularization step completely from the gradient updates. The moments $\mathbf{m}_t$ and $\mathbf{v}_t$ are computed purely using the data loss gradient. We apply the L2 decay *directly* to the weights at the end of the step:

$$ W_{t+1} = W_t - \alpha \lambda W_t - \frac{\alpha}{\sqrt{\hat{\mathbf{v}}_{t+1}} + \epsilon} \odot \hat{\mathbf{m}}_{t+1} $$

Where $\lambda$ represents the true weight decay coefficient. This restores clean, isotropic regularization.


### Taxonomy & Evolution of Optimizers

![Diagram 3](assets/lecture3_diagram_3.png)


## 3.8 NumPy Optimizers Implementation from Scratch

Below, we implement all discussed optimizers (SGD, Momentum, RMSProp, Adam, and AdamW) in raw NumPy and evaluate them navigating a highly non-convex, poor-conditioned mathematical loss landscape.

In [ ]:
import numpy as np

# Define a poor-conditioned, non-convex mathematical landscape: Rosenbrock function
# L(x, y) = (a - x)^2 + b * (y - x^2)^2. Global minimum is at (1, 1) inside a narrow valley.
def rosenbrock(W):
    x, y = W[0], W[1]
    return (1.0 - x)**2 + 100.0 * (y - x**2)**2

def rosenbrock_grad(W):
    x, y = W[0], W[1]
    dx = -2.0 * (1.0 - x) - 400.0 * x * (y - x**2)
    dy = 200.0 * (y - x**2)
    return np.array([dx, dy])

# --- OPTIMIZERS IMPLEMENTATION ---

def optimize_sgd(W_start, steps=2000, lr=0.001):
    W = W_start.copy()
    for _ in range(steps):
        grad = rosenbrock_grad(W)
        W -= lr * grad
    return W

def optimize_momentum(W_start, steps=2000, lr=0.001, rho=0.9):
    W = W_start.copy()
    v = np.zeros_like(W)
    for _ in range(steps):
        grad = rosenbrock_grad(W)
        v = rho * v + grad
        W -= lr * v
    return W

def optimize_rmsprop(W_start, steps=2000, lr=0.01, beta=0.99, epsilon=1e-8):
    W = W_start.copy()
    s = np.zeros_like(W)
    for _ in range(steps):
        grad = rosenbrock_grad(W)
        s = beta * s + (1.0 - beta) * (grad ** 2)
        W -= (lr / (np.sqrt(s) + epsilon)) * grad
    return W

def optimize_adam(W_start, steps=2000, lr=0.05, beta1=0.9, beta2=0.999, epsilon=1e-8):
    W = W_start.copy()
    m = np.zeros_like(W)
    v = np.zeros_like(W)
    for t in range(1, steps + 1):
        grad = rosenbrock_grad(W)
        m = beta1 * m + (1.0 - beta1) * grad
        v = beta2 * v + (1.0 - beta2) * (grad ** 2)
        
        # Bias correction
        m_hat = m / (1.0 - beta1 ** t)
        v_hat = v / (1.0 - beta2 ** t)
        
        W -= (lr / (np.sqrt(v_hat) + epsilon)) * m_hat
    return W

# Run optimizations starting from a challenging coordinate point far from the minimum
start_point = np.array([-1.5, 2.0])

sgd_result = optimize_sgd(start_point)
momentum_result = optimize_momentum(start_point)
rmsprop_result = optimize_rmsprop(start_point)
adam_result = optimize_adam(start_point)

print("--- Non-Convex Landscape Traversal Results ---")
print(f"Start Point: {start_point} | Target Minimum: [1.0, 1.0] (Loss: 0.0)")
print(f"SGD Final Coordinate: {sgd_result} | Loss: {rosenbrock(sgd_result):.6f}")
print(f"Momentum Final Coordinate: {momentum_result} | Loss: {rosenbrock(momentum_result):.6f}")
print(f"RMSProp Final Coordinate: {rmsprop_result} | Loss: {rosenbrock(rmsprop_result):.6f}")
print(f"Adam Final Coordinate: {adam_result} | Loss: {rosenbrock(adam_result):.6f}")


## 3.9 Summary & Transition to Schedulers

*   **Vanilla SGD** is unscalably slow for full datasets and is extremely vulnerable to poor conditioning (oscillations), local minima, and saddle points.
*   **Momentum** resolves valleys and flat saddle regions by accumulating directional speed (velocity $v$).
*   **RMSProp** adapts step sizes individually per coordinate, slowing down in steep canyons and accelerating along flat valleys.
*   **Adam** combines Momentum and RMSProp, and utilizes **bias correction** to ensure safe, stable updates during the initial steps.
*   **AdamW** decouples the L2 regularization step from the adaptive moments, maintaining isotropic weight decay.

---

**Up Next:** Even the most sophisticated optimizers struggle if their baseline step size $\alpha$ is poorly configured. In **Section 4**, we explore **Learning Rate Scheduling** and understand how to decay our step sizes over time to land precisely at the optimal landscape coordinates.

# Section 4: Learning Rate Scheduling

No matter how sophisticated our optimizer is, its ultimate success relies heavily on configuring the **Learning Rate** $\alpha$. In this section, we study why a fixed learning rate is sub-optimal and analyze modern scheduling strategies: **Step Decay**, **Cosine Decay**, **Linear Warmup**, and the **Linear Scaling Law**.

## 4.1 Importance of the Learning Rate $\alpha$

The learning rate $\alpha$ is the single most critical hyperparameter in deep learning. 

*   **$\alpha$ too high:** The weight updates are too large. The optimizer will overshoot the valley bottom, bounce back and forth, and eventually oscillate out of the loss landscape entirely (exploding loss).
*   **$\alpha$ too low:** The steps are tiny. Training is agonizingly slow, and the optimizer is highly likely to get permanently trapped in the first flat region, saddle point, or local minimum it encounters.
*   **$\alpha$ just right:** The model converges steadily and smoothly to a high-quality global minimum.

### Why a fixed learning rate is bad
At the beginning of training, weights are randomly initialized, and the loss is very high. We want a **large step size** to move quickly down steep mountainsides. 
However, as we approach the deep valley bottom, a large step size will cause the optimizer to bounce around the local minimum instead of descending into it. We need the step size to **shrink** over time so we can settle precisely at the optimal coordinates.

## 4.2 Schedulers (Step Decay, Cosine Decay, Warmup)

To adapt our step size during training, we define a **Learning Rate Scheduler** that updates $\alpha$ as a function of the training progress (time step $t$ or epoch $e$).

### 1. Step Decay
We reduce the learning rate by a multiplicative factor $\gamma$ (typically $0.1$ or $0.5$) after a fixed number of epochs $T$:

$$ \alpha_t = \alpha_0 \cdot \gamma^{\lfloor t / T \rfloor} $$

*   **Visual Profile:** The loss curve exhibits sudden, sharp drops at the boundary points where the learning rate is cut. This is standard when training ResNet architectures.

### 2. Cosine Decay
We smoothly decay the learning rate following the shape of a cosine curve from $\alpha_{max}$ down to a minimum target learning rate $\alpha_{min}$:

$$ \alpha_t = \alpha_{min} + \frac{1}{2} (\alpha_{max} - \alpha_{min}) \left( 1 + \cos\left( \frac{\pi t}{T_{max}} \right) \right) $$

where $T_{max}$ is the total scheduled training iterations.
*   **Visual Profile:** Highly smooth, continuous training progression without discontinuous drop steps. This is the default setup for state-of-the-art Transformer architectures.

### 3. Linear Warmup
At $t=0$, gradients are extremely noisy because weights are random. Large learning rates can cause the weights to destabilize immediately. 
To prevent this, we implement **Linear Warmup**: we start $\alpha$ at $0$ and linearly increase it to $\alpha_{max}$ over the first few thousand steps, before starting our decay schedule (e.g., Cosine Decay).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Let's programmatically simulate and visualize Step Decay, Cosine Decay, and Linear Warmup
total_steps = 1000
alpha_max = 0.1
alpha_min = 0.001

steps = np.arange(total_steps)

# 1. Step Decay (decay by 0.5 every 250 steps)
step_lrs = []
for t in steps:
    lr = alpha_max * (0.5 ** (t // 250))
    step_lrs.append(lr)

# 2. Cosine Decay
cosine_lrs = []
for t in steps:
    lr = alpha_min + 0.5 * (alpha_max - alpha_min) * (1.0 + np.cos(np.pi * t / total_steps))
    cosine_lrs.append(lr)

# 3. Cosine Decay with Linear Warmup (warmup over first 150 steps)
warmup_steps = 150
warmup_cosine_lrs = []
for t in steps:
    if t < warmup_steps:
        # Linear Warmup
        lr = (t / warmup_steps) * alpha_max
    else:
        # Cosine Decay for the remaining steps
        decay_ratio = (t - warmup_steps) / (total_steps - warmup_steps)
        lr = alpha_min + 0.5 * (alpha_max - alpha_min) * (1.0 + np.cos(np.pi * decay_ratio))
    warmup_cosine_lrs.append(lr)

# Plot the learning rate curves
plt.figure(figsize=(10, 5))
plt.plot(steps, step_lrs, label='Step Decay', linewidth=2)
plt.plot(steps, cosine_lrs, label='Cosine Decay', linewidth=2)
plt.plot(steps, warmup_cosine_lrs, label='Cosine Decay with Linear Warmup', linewidth=2, linestyle='--')
plt.title("Comparison of Learning Rate Schedules", fontsize=14)
plt.xlabel("Training Steps", fontsize=12)
plt.ylabel("Learning Rate ($\alpha$)", fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.show()


## 4.3 Batch Size & The Linear Scaling Law

When scaling model training to multiple GPUs, we often increase the **Batch Size** $B$ to maximize hardware efficiency. However, increasing $B$ requires adjusting the learning rate $\alpha$.

### The Linear Scaling Law
The **Linear Scaling Law** (empirically validated by Goyal et al. in 2017) states:
> **If we scale the batch size $B$ by a factor of $k$, we must scale the initial learning rate $\alpha$ by the exact same factor $k$.**
>
> $$ B \leftarrow k \cdot B \implies \alpha_0 \leftarrow k \cdot \alpha_0 $$

### Statistical Intuition
Why is this law so effective?
*   **Variance Reduction:** When we compute the gradient over a mini-batch of size $B$, the gradient vector has a certain variance. When we increase the batch size to $k \cdot B$, the variance of our gradient estimate is reduced by a factor of $k$ due to the **Central Limit Theorem**.
*   **Higher Confidence:** Since our gradient estimate is much less noisy and far more accurate, we can highly confidentially take a **much larger step size** ($k \cdot \alpha_0$) in that direction without the risk of destabilization.

# Section 5: Second-Order Optimization (Brief Overview)

All the optimization algorithms we have discussed so far (SGD, Momentum, RMSProp, Adam) are **First-Order Optimizers** because they rely purely on the first derivative (the gradient vector) to approximate the loss landscape linearly.

In this concluding section, we briefly explore **Second-Order Optimization** (which incorporates second derivatives) and analyze why it remains impractical for training deep neural networks.

## 5.1 Second-Order Approximations & Newton's Method

First-order optimizers construct a local linear (tangent) approximation of the loss landscape. Second-order optimizers use a quadratic Taylor series expansion to fit a local multi-dimensional paraboloid to the landscape.

### Newton's Method
The standard second-order optimization update is **Newton's Method**:

$$ W_{t+1} = W_t - H^{-1} \nabla_W L(W_t) $$

Where:
*   $\nabla_W L(W_t) \in \mathbb{R}^{D \times 1}$ is the standard gradient vector.
*   $H \in \mathbb{R}^{D \times D}$ is the **Hessian Matrix** containing the second-order partial derivatives of the loss with respect to all parameter combinations:

$$ H_{j,k} = \frac{\partial^2 L}{\partial W_j \partial W_k} $$

> [!IMPORTANT]
> **The No-Hyperparameter Miracle:** Notice that vanilla Newton's Method has **no learning rate hyperparameter $\alpha$!** Because it fits a local quadratic bowl, it mathematically solves for the exact step length required to jump straight to the local minimum of that quadratic approximation.

## 5.2 Why Newton's Method Fails in Deep Learning

If Newton's Method can jump straight to local minima with zero hyperparameter tuning, why do we never use it to train deep neural networks? 

There are three catastrophic roadblocks:

### 1. Memory Complexity (O(D^2))
If a neural network has $D$ parameters, the Hessian matrix contains $D \times D$ elements.
*   For a relatively small model with $D = 100$ million parameters, the Hessian contains $100,000,000 \times 100,000,000 = 10^{16}$ floating-point values.
*   Storing this single matrix requires **40,000 Terabytes** of GPU memory! This is physically impossible.

### 2. Time Complexity (O(D^3))
Newton's Method requires computing the inverse of the Hessian matrix ($H^{-1}$). Inverting a matrix of size $D \times D$ has a cubic computational complexity of $O(D^3)$. 
*   For $D = 100$ million parameters, computing this inverse would take months or years of calculation *for a single update step*.

### 3. Saddle Point Vulnerability
Newton's Method actively seeks out points where the gradient is exactly $0$. It does not distinguish between local minima, local maxima, and saddle points.
*   As established in Section 3, high-dimensional neural network loss landscapes are dominated by **saddle points**, not local minima.
*   Newton's Method will actively steer the weights straight towards saddle points and get stuck there, whereas first-order momentum optimizers successfully roll past them.

## 5.3 Final Summary & Lecture Takeaways

We have successfully constructed a complete, comprehensive university-level study note for CS231n Lecture 3. Here is the holistic summary of our learning progression:

1.  **Objective Formulation:** The loss function balances data alignment ($L_{data}$) and model generalization ($R(W)$ via Occam's Razor).
2.  **Navigation (The Gradient):** Gaining local slope information ($
abla_W L$) is the only computationally viable way to guide parameter updates.
3.  **Optimization Engines:** We start with SGD, identify its geological failure modes, and build: 
    *   *Momentum* (dampens oscillations, clears saddle points),
    *   *RMSProp* (individual coordinate learning rate adaptation),
    *   *Adam/AdamW* (combining first and second moments with bias corrections and decoupled weight decay).
4.  **Step Configuration:** Step size $\alpha$ must be schedule-decayed (Step/Cosine) over time. Scaling Batch Sizes requires scaling Learning Rates proportionally (Linear Scaling Law).
5.  **Complexity Limits:** Second-order methods (Newton's Method) are mathematically beautiful but computationally prohibitive for large-scale parametric machine learning.

---